In [10]:
# %% [markdown]
# # Shared Config

# %%
import os
import sys
import json
import time
import random
import importlib
from pathlib import Path
from argparse import Namespace

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ---------------------------------------------------------------------
# Shared project paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design")
BENCHMARK_ROOT = PROJECT_ROOT / "utils" / "benchmarks"
TIMING_RESULTS_ROOT = BENCHMARK_ROOT / "timing_results"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------
# Shared experiment grid
# ---------------------------------------------------------------------
SEEDS = [0]
N_QUBITS = [4, 6, 8]

DATASETS = {
    "iris": (61, 4),
    "wine": (187, 13),
    "diabetes": (37, 8),
}

# ---------------------------------------------------------------------
# Shared runtime/training config
# ---------------------------------------------------------------------
BATCH_SIZE = 32
USE_GPU = False
VERBOSE_ON_FIT = False
DO_PCA = True

torch.set_num_threads(1)

# ---------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def iter_experiments():
    for seed in SEEDS:
        for n_qubits in N_QUBITS:
            for dataset_name, (dataset_id, n_features) in DATASETS.items():
                if n_qubits <= n_features:
                    yield {
                        "seed": seed,
                        "n_qubits": n_qubits,
                        "dataset_name": dataset_name,
                        "dataset_id": dataset_id,
                        "n_features": n_features,
                    }


def make_timings_dir(method_name: str, dataset_name: str, seed: int, n_qubits: int) -> Path:
    return TIMING_RESULTS_ROOT / method_name / dataset_name / f"seed_{seed}" / f"{n_qubits}qubits"

# QuantumDARTS

In [6]:
# %% [markdown]
# # QuantumDARTS

# %%
from QuantumDARTS.quantum_darts_model import QuantumDARTSModel
from QuantumDARTS.trainer import DARTSTrainer
from QuantumDARTS.utils import (
    GATE_POOL,
    load_mnist_data,
    load_openml_data,
    make_simple_multiclass_data,
)

QDARTS_CONFIG = {
    "learning_rate_arch": 0.01,
    "learning_rate_rot": 0.01,
    "perform_intermediate_eval": True,
    "eval_every": 2,
    "retraining_steps": 30,
    "retraining_lr": 0.01,
    "pilot_search_epochs": 5,
    "target_full_search_epochs": 10,
}


def quantumdarts_num_layers(n_qubits: int) -> int:
    return int(24 / n_qubits)


def build_darts_dataloaders(dataset_id, n_qubits, seed):
    if dataset_id == 0:
        n_classes = 3
        X_train, y_train, X_val, y_val, X_test, y_test = make_simple_multiclass_data(
            n_features=n_qubits,
            n_classes=n_classes,
            n_samples=500,
            cluster_std=0.01,
            random_state=42,
        )
    elif dataset_id == 1:
        X_train, y_train, X_val, y_val, X_test, y_test = load_mnist_data(
            n_features=n_qubits
        )
        n_classes = 2
    elif dataset_id in [61, 187, 37, 1510]:
        X_train, y_train, X_val, y_val, X_test, y_test = load_openml_data(
            dataset_id, n_features=n_qubits, seed=seed
        )
        n_classes = len(np.unique(y_train))
    else:
        raise ValueError(f"Unsupported dataset_id {dataset_id}")

    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    val_dataset = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    test_dataset = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long),
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader, n_classes


def run_timed_quantumdarts(dataset_name, dataset_id, n_qubits, seed, timings_dir):
    n_layers = quantumdarts_num_layers(n_qubits)

    _, val_dataset, _, train_loader, val_loader, test_loader, n_classes = build_darts_dataloaders(
        dataset_id, n_qubits, seed
    )

    model = QuantumDARTSModel(
        n_qubits=n_qubits,
        n_layers=n_layers,
        gate_pool=GATE_POOL,
        n_classes=n_classes,
    )

    trainer = DARTSTrainer(
        model=model,
        train_loader=train_loader,
        validation_data=val_dataset,
        lr_arch=QDARTS_CONFIG["learning_rate_arch"],
        lr_rot=QDARTS_CONFIG["learning_rate_rot"],
    )

    search_t0 = time.perf_counter()
    trainer.search(
        num_epochs=QDARTS_CONFIG["pilot_search_epochs"],
        perform_eval=QDARTS_CONFIG["perform_intermediate_eval"],
        eval_every=QDARTS_CONFIG["eval_every"],
    )
    measured_pilot_search_time_sec = time.perf_counter() - search_t0

    estimated_full_search_time_sec = (
        measured_pilot_search_time_sec / QDARTS_CONFIG["pilot_search_epochs"]
    ) * QDARTS_CONFIG["target_full_search_epochs"]

    derive_t0 = time.perf_counter()
    _final_circuit = trainer.derive_best_circuit()
    derive_time_sec = time.perf_counter() - derive_t0

    reported_search_time_sec = estimated_full_search_time_sec + derive_time_sec

    train_t0 = time.perf_counter()
    _ = trainer.final_evaluation(
        train_loader,
        val_loader,
        test_loader,
        QDARTS_CONFIG["retraining_steps"],
        QDARTS_CONFIG["retraining_lr"],
    )
    measured_final_train_time_sec = time.perf_counter() - train_t0

    reported_train_time_sec = measured_final_train_time_sec
    reported_total_time_sec = reported_search_time_sec + reported_train_time_sec

    output = {
        "method": "QuantumDARTS",
        "dataset": dataset_name,
        "seed": int(seed),
        "qubits": int(n_qubits),
        "runtime_table_sec": {
            "search": float(reported_search_time_sec),
            "train": float(reported_train_time_sec),
            "total": float(reported_total_time_sec),
        },
        "timings_sec": {
            "measured_pilot_search_time_sec": float(measured_pilot_search_time_sec),
            "estimated_full_search_time_sec": float(estimated_full_search_time_sec),
            "derive_time_sec": float(derive_time_sec),
            "measured_final_train_time_sec": float(measured_final_train_time_sec),
        },
        "pilot_config": {
            "pilot_search_epochs": int(QDARTS_CONFIG["pilot_search_epochs"]),
        },
        "full_config": {
            "target_full_search_epochs": int(QDARTS_CONFIG["target_full_search_epochs"]),
            "retraining_steps": int(QDARTS_CONFIG["retraining_steps"]),
        },
    }

    os.makedirs(timings_dir, exist_ok=True)
    out_fp = os.path.join(timings_dir, "final_results.json")
    with open(out_fp, "w") as f:
        json.dump(output, f, indent=2)

    print(f"[INFO] Saved timing-only results to {out_fp}")


if __name__ == "__main__":
    for exp in iter_experiments():
        set_seed(exp["seed"])
        run_timed_quantumdarts(
            dataset_name=exp["dataset_name"],
            dataset_id=exp["dataset_id"],
            n_qubits=exp["n_qubits"],
            seed=exp["seed"],
            timings_dir=str(make_timings_dir("QuantumDARTS", exp["dataset_name"], exp["seed"], exp["n_qubits"])),
        )

[env] Loading OpenML data for dataset ID: 61
Dataset name: iris
Total features: 4, Dropped Categorical features: 0, Remaining features: 4
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.


Searching:  40%|████      | 2/5 [03:48<06:26, 128.90s/it, acc=0.3448, loss=1.4927]

epoch=2 pred_hist=[24, 5, 0] val_acc=0.2759


Searching:  80%|████████  | 4/5 [07:23<02:04, 124.05s/it, acc=0.3793, loss=1.7582]

epoch=4 pred_hist=[21, 7, 1] val_acc=0.2759


...

Epoch 27/30:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 28/30:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 29/30:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 30/30:   0%|          | 0/15 [00:00<?, ?it/s]

Test Results: Loss=0.6756, Acc=0.5519, F1=0.0000
[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/QuantumDARTS/diabetes/seed_4/8qubits/final_results.json


# TF-QAS

In [9]:
# %% [markdown]
# # TF-QAS

# %%
from utils.data.preprocessing import data_pipeline
from utils.nn.QuantumNN import QuantumNN
from utils.ansatze.HierarchicalGatewise import HierarchicalGatewiseAnsatz
from utils.ansatze.HierarchicalLayerwise import HierarchicalLayerwiseAnsatz

exp_module = importlib.import_module("TF-QAS.expressibility_proxy")
path_module = importlib.import_module("TF-QAS.path_proxy")

calculate_expressibility_proxy = exp_module.calculate_expressibility_proxy
calculate_path_proxy = path_module.calculate_path_proxy

TFQAS_CONFIG = {
    "quantum_lr": 0.05,
    "pilot_s": 200,
    "pilot_r": 20,
    "pilot_k": 5,
    "pilot_epochs_quantum": 5,
    "full_s": 50000,
    "full_r": 5000,
    "full_k": 50,
    "full_epochs_quantum": 30,
    "mode": "gate",  # or "layer"
}


def run_tf_qas_timed(ansatze_class, ansatz_args, S, R, K):
    qas_t0 = time.perf_counter()

    initial_circuits = []
    for i in range(S):
        ansatz = ansatze_class(**ansatz_args, seed=i)
        initial_circuits.append({"seed": i, "ansatz": ansatz})

    for circuit_data in initial_circuits:
        path_count = calculate_path_proxy(circuit_data["ansatz"].get_ansatz())
        circuit_data["path_count"] = path_count

    sorted_by_path = sorted(initial_circuits, key=lambda x: x["path_count"], reverse=True)
    top_R_circuits = sorted_by_path[:R]

    for circuit_data in top_R_circuits:
        expressibility = calculate_expressibility_proxy(
            circuit_data["ansatz"].get_ansatz(),
            n_samples=200,
        )
        circuit_data["expressibility"] = expressibility

    sorted_by_expressibility = sorted(
        top_R_circuits,
        key=lambda x: x["expressibility"],
        reverse=True,
    )
    top_K_circuits = sorted_by_expressibility[:K]

    qas_time_sec = time.perf_counter() - qas_t0
    return top_K_circuits, qas_time_sec


def train_candidate_once(
    circuit,
    input_dim,
    output_dim,
    train_loader_q,
    val_loader_q,
    test_loader_q,
    epochs,
    seed,
):
    set_seed(seed)

    model_q = QuantumNN(
        ansatz=circuit.get_ansatz(),
        n_qubits=input_dim,
        num_classes=output_dim,
        use_gpu=USE_GPU,
        gradient_method="guided_spsa",
    )

    optimizer_q = optim.Adam(model_q.parameters(), lr=TFQAS_CONFIG["quantum_lr"])
    scheduler_q = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_q,
        mode="min",
        factor=0.3,
        patience=max(1, epochs // 5),
        min_lr=1e-4,
        verbose=False,
    )

    t0 = time.perf_counter()
    model_q.fit(
        train_loader_q,
        val_loader_q,
        epochs=epochs,
        optimizer=optimizer_q,
        scheduler=scheduler_q,
        verbose=VERBOSE_ON_FIT,
        eval_every=2,
    )
    train_time_sec = time.perf_counter() - t0

    eval_output_q = model_q.evaluate(test_loader_q, verbose=False)
    metrics = {
        "test_loss": eval_output_q[0],
        "test_acc": eval_output_q[1],
        "test_prec": eval_output_q[2],
        "test_rec": eval_output_q[3],
        "test_f1": eval_output_q[4],
    }
    return metrics, train_time_sec


def run_timed_tf_qas(dataset_name, openml_dataset_id, n_qubits, seed, timings_dir):
    _, quantum_data, input_dim, output_dim, _ = data_pipeline(
        openml_dataset_id,
        batch_size=BATCH_SIZE,
        do_pca=DO_PCA,
        use_gpu=USE_GPU,
        n_components=n_qubits,
        seed=seed,
    )

    (
        x_train_q,
        x_val_q,
        x_test_q,
        y_train_q,
        y_val_q,
        y_test_q,
        train_loader_q,
        val_loader_q,
        test_loader_q,
    ) = quantum_data

    ansatz_args = {"n_qubits": n_qubits}
    if TFQAS_CONFIG["mode"] == "gate":
        ansatze_class = HierarchicalGatewiseAnsatz
        ansatz_args["n_gates"] = 24
        ansatz_args["temporal_bias"] = 0.75
        ansatz_args["spatial_bias"] = 1.0
    elif TFQAS_CONFIG["mode"] == "layer":
        ansatze_class = HierarchicalLayerwiseAnsatz
        ansatz_args["depth"] = 4
    else:
        raise ValueError("mode must be 'gate' or 'layer'")

    candidate_circuits, measured_qas_stage_time_sec = run_tf_qas_timed(
        ansatze_class,
        ansatz_args,
        TFQAS_CONFIG["pilot_s"],
        TFQAS_CONFIG["pilot_r"],
        TFQAS_CONFIG["pilot_k"],
    )

    candidate_training_time_sec = 0.0
    best_test_f1 = -1.0
    best_circuit = None

    for ansatz_data in candidate_circuits:
        circuit = ansatz_data["ansatz"]
        metrics, this_train_time_sec = train_candidate_once(
            circuit=circuit,
            input_dim=input_dim,
            output_dim=output_dim,
            train_loader_q=train_loader_q,
            val_loader_q=val_loader_q,
            test_loader_q=test_loader_q,
            epochs=TFQAS_CONFIG["pilot_epochs_quantum"],
            seed=seed,
        )
        candidate_training_time_sec += this_train_time_sec

        if metrics["test_f1"] > best_test_f1:
            best_test_f1 = metrics["test_f1"]
            best_circuit = circuit

    estimated_qas_stage_time_sec = (
        measured_qas_stage_time_sec * (TFQAS_CONFIG["full_s"] / TFQAS_CONFIG["pilot_s"])
    )
    estimated_candidate_training_time_sec = (
        candidate_training_time_sec
        * (TFQAS_CONFIG["full_k"] / TFQAS_CONFIG["pilot_k"])
        * (TFQAS_CONFIG["full_epochs_quantum"] / TFQAS_CONFIG["pilot_epochs_quantum"])
    )

    reported_search_time_sec = (
        estimated_qas_stage_time_sec + estimated_candidate_training_time_sec
    )

    _, measured_final_retrain_time_sec = train_candidate_once(
        circuit=best_circuit,
        input_dim=input_dim,
        output_dim=output_dim,
        train_loader_q=train_loader_q,
        val_loader_q=val_loader_q,
        test_loader_q=test_loader_q,
        epochs=TFQAS_CONFIG["full_epochs_quantum"],
        seed=seed,
    )

    reported_train_time_sec = measured_final_retrain_time_sec
    reported_total_time_sec = reported_search_time_sec + reported_train_time_sec

    output_data = {
        "method": "TF-QAS",
        "dataset": dataset_name,
        "seed": int(seed),
        "qubits": int(n_qubits),
        "runtime_table_sec": {
            "search": float(reported_search_time_sec),
            "train": float(reported_train_time_sec),
            "total": float(reported_total_time_sec),
        },
        "timings_sec": {
            "measured_qas_stage_time_sec": float(measured_qas_stage_time_sec),
            "candidate_training_time_sec_pilot": float(candidate_training_time_sec),
            "estimated_qas_stage_time_sec": float(estimated_qas_stage_time_sec),
            "estimated_candidate_training_time_sec": float(estimated_candidate_training_time_sec),
            "measured_final_retrain_time_sec": float(measured_final_retrain_time_sec),
        },
        "pilot_config": {
            "S": int(TFQAS_CONFIG["pilot_s"]),
            "R": int(TFQAS_CONFIG["pilot_r"]),
            "K": int(TFQAS_CONFIG["pilot_k"]),
            "epochs_quantum": int(TFQAS_CONFIG["pilot_epochs_quantum"]),
        },
        "full_config": {
            "S": int(TFQAS_CONFIG["full_s"]),
            "R": int(TFQAS_CONFIG["full_r"]),
            "K": int(TFQAS_CONFIG["full_k"]),
            "epochs_quantum": int(TFQAS_CONFIG["full_epochs_quantum"]),
        },
    }

    os.makedirs(timings_dir, exist_ok=True)
    out_fp = os.path.join(timings_dir, "final_results.json")
    with open(out_fp, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"[INFO] Saved timing-only results to {out_fp}")


if __name__ == "__main__":
    for exp in iter_experiments():
        set_seed(exp["seed"])
        run_timed_tf_qas(
            dataset_name=exp["dataset_name"],
            openml_dataset_id=exp["dataset_id"],
            n_qubits=exp["n_qubits"],
            seed=exp["seed"],
            timings_dir=str(make_timings_dir("TF-QAS", exp["dataset_name"], exp["seed"], exp["n_qubits"])),
        )

Dataset name: iris
Total features: 4, Dropped Categorical features: 0, Remaining features: 4
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Initialized QuantumNN on device: cpu


...

Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/TF-QAS/diabetes/seed_2/8qubits/final_results.json


# BenchRL-QAS

In [11]:
# %% [markdown]
# # BenchRL-QAS

# %%
NOTEBOOK_DIR = Path.cwd()
BENCHMARKS_DIR = NOTEBOOK_DIR
RL_VQC_DIR = BENCHMARKS_DIR / "RL-QAS-TPPO" / "VQC"
RL_CONFIG_DIR = RL_VQC_DIR / "configuration_files" / "TPPO"
BENCHRL_TIMING_ROOT = TIMING_RESULTS_ROOT / "BenchRL-QAS-TPPO"

for p in [PROJECT_ROOT, RL_VQC_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from vqc_helpers import get_config
from environment_VQC import CircuitEnv
import agents

from qiskit.circuit import QuantumCircuit as QkCircuit, Parameter

BENCHRL_CONFIG = {
    "pilot_episodes": 10,
    "full_episodes": 200,
    "final_train_epochs": 10,
    "final_train_lr": 0.05,
}


def modify_state(state, env, conf, device):
    if not torch.is_tensor(state):
        state = torch.tensor(state, dtype=torch.double, device=device)
    if conf["agent"]["en_state"]:
        if env.prev_energy is None:
            raise ValueError("env.prev_energy is None; ensure CircuitEnv.step() sets it correctly")
        prev_energy = torch.tensor(float(env.prev_energy), dtype=torch.double, device=device).unsqueeze(0)
        state = torch.cat((state, prev_energy))
    if "threshold_in_state" in conf["agent"].keys() and conf["agent"]["threshold_in_state"]:
        done_threshold = torch.tensor([env.done_threshold], dtype=torch.double, device=device)
        state = torch.cat((state, done_threshold))
    return state


def modify_state_like_training(state, env, conf, device):
    if conf["agent"].get("en_state", 0):
        state = torch.cat((state, torch.tensor(env.prev_energy, dtype=torch.float, device=device).view(1)))
    if conf["agent"].get("threshold_in_state", 0):
        state = torch.cat((state, torch.tensor(env.done_threshold, dtype=torch.float, device=device).view(1)))
    return state


def greedy_eval_episode(env, agent, device, conf):
    state = env.reset().clone().detach().to(dtype=torch.double, device=device)
    state = modify_state_like_training(state, env, conf, device)

    for _ in range(env.num_layers + 1):
        illegal = env.illegal_action_new()
        with torch.no_grad():
            logits = agent.policy(state)
            probs = torch.softmax(logits, dim=-1)
            if illegal is not None:
                probs[illegal] = 0
                probs = probs / probs.sum()
            action = torch.argmax(probs).item()

        next_state, reward, done = env.step(agent.translate[action])
        next_state = next_state.clone().detach().to(dtype=torch.double, device=device)
        next_state = modify_state_like_training(next_state, env, conf, device)
        state = next_state

        if done:
            break

    from qulacs import ParametricQuantumCircuit as PQC

    if hasattr(env, "get_parametric_circuit"):
        try:
            circ = env.get_parametric_circuit()
            if circ is not None:
                return circ
        except Exception:
            pass

    for name in ("ansatz", "circuit", "parametric_circuit", "pq_circuit"):
        if hasattr(env, name):
            circ = getattr(env, name)
            try:
                if isinstance(circ, PQC):
                    return circ
            except Exception:
                return circ

    if hasattr(env, "make_circuit"):
        try:
            circ = env.make_circuit()
            if circ is not None:
                setattr(env, "ansatz", circ)
                return circ
        except Exception:
            pass

    try:
        for _, v in env.__dict__.items():
            if isinstance(v, PQC):
                return v
    except Exception:
        pass

    return None


def one_episode(episode_no, env, agent, episodes, conf):
    t0 = time.time()
    agent.saver.get_new_episode("train", episode_no)
    state = env.reset()
    agent.saver.stats_file["train"][episode_no]["done_threshold"] = env.done_threshold

    state = modify_state(state, env, conf, agent.device)

    done = False
    for _ in range(env.num_layers + 1):
        action, log_prob = agent.act(state)
        next_state, reward, done = env.step(agent.translate[action])
        agent.rewards.append(reward.item() if torch.is_tensor(reward) else reward)

        agent.saver.stats_file["train"][episode_no]["actions"].append(action)
        agent.saver.stats_file["train"][episode_no]["errors"].append(env.error)
        agent.saver.stats_file["train"][episode_no]["errors_test"].append(env.error_test)
        agent.saver.stats_file["train"][episode_no]["rewards"].append(env.current_reward)
        agent.saver.stats_file["train"][episode_no]["time"].append(time.time() - t0)

        next_state = modify_state(next_state, env, conf, agent.device)
        state = next_state

        if done:
            break

    agent.update()
    agent.saver.validate_stats(episode_no, "train")


def train(agent, env, episodes, seed, output_path, threshold, conf):
    for e in range(episodes):
        one_episode(e, env, agent, episodes, conf)


class Saver:
    def __init__(self, results_path, experiment_seed):
        self.stats_file = {"train": {}, "test": {}}
        self.exp_seed = experiment_seed
        self.rpath = results_path

    def get_new_episode(self, mode, episode_no):
        if mode == "train":
            self.stats_file[mode][episode_no] = {
                "loss_policy": [],
                "loss_value": [],
                "actions": [],
                "errors": [],
                "errors_test": [],
                "done_threshold": 0,
                "bond_distance": 0,
                "nfev": [],
                "opt_ang": [],
                "time": [],
                "rewards": [],
            }

    def validate_stats(self, episode, mode):
        assert len(self.stats_file[mode][episode]["actions"]) == len(self.stats_file[mode][episode]["errors"])


def convert_qulacs_to_qiskit(qulacs_circuit):
    num_qubits = qulacs_circuit.get_qubit_count()
    qiskit_qc = QkCircuit(num_qubits)
    param_index = 0

    gate_count = qulacs_circuit.get_gate_count()
    for i in range(gate_count):
        gate = qulacs_circuit.get_gate(i)
        name = gate.get_name()
        targets = gate.get_target_index_list()
        controls = gate.get_control_index_list()

        if name == "CNOT":
            qiskit_qc.cx(controls[0], targets[0])
        elif name == "ParametricRY":
            theta = Parameter(f"θ{param_index}")
            qiskit_qc.ry(theta, targets[0])
            param_index += 1
        elif name == "ParametricRX":
            theta = Parameter(f"θ{param_index}")
            qiskit_qc.rx(theta, targets[0])
            param_index += 1
        elif name == "ParametricRZ":
            theta = Parameter(f"θ{param_index}")
            qiskit_qc.rz(theta, targets[0])
            param_index += 1
        elif name == "X":
            qiskit_qc.x(targets[0])
        elif name == "H":
            qiskit_qc.h(targets[0])
        else:
            print(f"Warning: Skipping unsupported gate for conversion: {name}")

    return qiskit_qc


def build_quantum_dataloaders(openml_dataset_id: int, n_qubits: int, seed: int):
    _, quantum_data, input_dim, output_dim, _ = data_pipeline(
        openml_dataset_id=openml_dataset_id,
        n_components=n_qubits,
        do_pca=DO_PCA,
        batch_size=BATCH_SIZE,
        seed=seed,
    )

    (
        x_train_q,
        x_val_q,
        x_test_q,
        y_train_q,
        y_val_q,
        y_test_q,
        train_loader_q,
        val_loader_q,
        test_loader_q,
    ) = quantum_data

    return {
        "input_dim": input_dim,
        "output_dim": output_dim,
        "train_loader": train_loader_q,
        "val_loader": val_loader_q,
        "test_loader": test_loader_q,
    }


def final_retrain_selected_circuit(qiskit_circuit, openml_dataset_id: int, n_qubits: int, seed: int):
    data = build_quantum_dataloaders(openml_dataset_id, n_qubits, seed)

    model_q = QuantumNN(
        ansatz=qiskit_circuit,
        n_qubits=data["input_dim"],
        num_classes=data["output_dim"],
        use_gpu=USE_GPU,
        gradient_method="guided_spsa",
        seed=seed,
    )

    optimizer_q = optim.Adam(model_q.parameters(), lr=BENCHRL_CONFIG["final_train_lr"])
    scheduler_q = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_q,
        mode="min",
        factor=0.3,
        patience=max(1, BENCHRL_CONFIG["final_train_epochs"] // 5),
        min_lr=1e-4,
        verbose=False,
    )

    t0 = time.perf_counter()
    model_q.fit(
        data["train_loader"],
        data["val_loader"],
        epochs=BENCHRL_CONFIG["final_train_epochs"],
        optimizer=optimizer_q,
        scheduler=scheduler_q,
        verbose=VERBOSE_ON_FIT,
        eval_every=2,
    )
    final_train_time_sec = time.perf_counter() - t0

    return final_train_time_sec


def run_timed_rl_qas(dataset_name, openml_dataset_id, n_qubits, seed, timings_dir):
    config_stem = f"{dataset_name}_coblya_{n_qubits}q_VQC"
    config_path = RL_CONFIG_DIR / f"{config_stem}.cfg"

    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")

    args_dict = {
        "config": str(config_path),
        "output_fp": str(BENCHRL_TIMING_ROOT / dataset_name / f"seed_{seed}" / f"{n_qubits}qubits"),
        "experiment_name": "TPPO",
    }
    args = Namespace(**args_dict)

    results_path = Path(args.output_fp)
    results_path.mkdir(parents=True, exist_ok=True)

    device = torch.device("cpu")
    conf = get_config(args.experiment_name, args.config)

    conf["openml"]["dataset_id"] = openml_dataset_id
    conf["env"]["num_qubits"] = n_qubits

    environment = CircuitEnv(conf, seed, device=device)

    initial_state = environment.reset()
    base_state_size = initial_state.shape[0]
    effective_state_size = base_state_size
    if conf["agent"]["en_state"]:
        effective_state_size += 1
    if "threshold_in_state" in conf["agent"].keys() and conf["agent"]["threshold_in_state"]:
        effective_state_size += 1

    agent = agents.__dict__[conf["agent"]["agent_type"]].__dict__[conf["agent"]["agent_class"]](
        conf, environment.action_size, effective_state_size, device
    )
    agent.saver = Saver(str(results_path), seed)

    search_t0 = time.perf_counter()
    train(
        agent,
        environment,
        BENCHRL_CONFIG["pilot_episodes"],
        seed,
        str(results_path),
        conf["env"]["accept_err"],
        conf,
    )
    measured_pilot_rl_training_time_sec = time.perf_counter() - search_t0

    extract_t0 = time.perf_counter()
    qulacs_circuit = greedy_eval_episode(environment, agent, device, conf)
    greedy_circuit_extraction_time_sec = time.perf_counter() - extract_t0

    if qulacs_circuit is None:
        raise RuntimeError("Could not retrieve circuit from greedy_eval_episode.")

    qiskit_circuit = convert_qulacs_to_qiskit(qulacs_circuit)

    estimated_full_rl_training_time_sec = (
        measured_pilot_rl_training_time_sec
        * (BENCHRL_CONFIG["full_episodes"] / BENCHRL_CONFIG["pilot_episodes"])
    )

    reported_search_time_sec = (
        estimated_full_rl_training_time_sec + greedy_circuit_extraction_time_sec
    )

    measured_final_train_time_sec = final_retrain_selected_circuit(
        qiskit_circuit=qiskit_circuit,
        openml_dataset_id=openml_dataset_id,
        n_qubits=n_qubits,
        seed=seed,
    )

    reported_train_time_sec = measured_final_train_time_sec
    reported_total_time_sec = reported_search_time_sec + reported_train_time_sec

    output_data = {
        "method": "BenchRL-QAS-TPPO",
        "dataset": dataset_name,
        "seed": int(seed),
        "qubits": int(n_qubits),
        "runtime_table_sec": {
            "search": float(reported_search_time_sec),
            "train": float(reported_train_time_sec),
            "total": float(reported_total_time_sec),
        },
        "timings_sec": {
            "measured_pilot_rl_training_time_sec": float(measured_pilot_rl_training_time_sec),
            "estimated_full_rl_training_time_sec": float(estimated_full_rl_training_time_sec),
            "greedy_circuit_extraction_time_sec": float(greedy_circuit_extraction_time_sec),
            "measured_final_train_time_sec": float(measured_final_train_time_sec),
        },
        "pilot_config": {
            "episodes": int(BENCHRL_CONFIG["pilot_episodes"]),
        },
        "full_config": {
            "episodes": int(BENCHRL_CONFIG["full_episodes"]),
            "final_train_epochs": int(BENCHRL_CONFIG["final_train_epochs"]),
        },
    }

    os.makedirs(timings_dir, exist_ok=True)
    out_fp = os.path.join(timings_dir, "final_results.json")
    with open(out_fp, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"[INFO] Saved timing-only results to {out_fp}")


if __name__ == "__main__":
    for exp in iter_experiments():
        set_seed(exp["seed"])
        run_timed_rl_qas(
            dataset_name=exp["dataset_name"],
            openml_dataset_id=exp["dataset_id"],
            n_qubits=exp["n_qubits"],
            seed=exp["seed"],
            timings_dir=str(make_timings_dir("BenchRL-QAS-TPPO", exp["dataset_name"], exp["seed"], exp["n_qubits"])),
        )

[env] Loading OpenML data for dataset ID: 61
Dataset name: iris
Total features: 4, Dropped Categorical features: 0, Remaining features: 4
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.
[env] OpenML splits → train (88, 4), test (30, 4)
Dataset name: iris
Total features: 4, Dropped Categorical features: 0, Remaining features: 4
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/BenchRL-QAS-TPPO/iris/seed_0/4qubits/final_results.json
[env] Loading OpenML data for dataset ID: 187
Dataset name: wine
Total features: 13, Dropped Categorical features: 0, Remaining features: 13
Classical data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
[env] OpenML splits → train (106, 4), test (36, 4)
Dataset name: wine
Total features: 13, Dropped Categorical features: 0, Remaining features: 13
Classical data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/BenchRL-QAS-TPPO/wine/seed_0/4qubits/final_results.json
[env] Loading OpenML data for dataset ID: 37
Dataset name: diabetes
Total features: 8, Dropped Categorical features: 0, Remaining features: 8
Classical data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
[env] OpenML splits → train (460, 4), test (154, 4)
Dataset name: diabetes
Total features: 8, Dropped Categorical features: 0, Remaining features: 8
Classical data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 4 dimensions using PCA.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


...

[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/BenchRL-QAS-TPPO/diabetes/seed_0/6qubits/final_results.json
[env] Loading OpenML data for dataset ID: 187
Dataset name: wine
Total features: 13, Dropped Categorical features: 0, Remaining features: 13
Classical data reduced to 8 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 8 dimensions using PCA.
Configuring DataLoader for CPU.
[env] OpenML splits → train (106, 8), test (36, 8)
Dataset name: wine
Total features: 13, Dropped Categorical features: 0, Remaining features: 13
Classical data reduced to 8 dimensions using PCA.
Configuring DataLoader for CPU.
Quantum data reduced to 8 dimensions using PCA.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/BenchRL-QAS-TPPO/wine/seed_0/8qubits/final_results.json
[env] Loading OpenML data for dataset ID: 37
Dataset name: diabetes
Total features: 8, Dropped Categorical features: 0, Remaining features: 8
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.
[env] OpenML splits → train (460, 8), test (154, 8)
Dataset name: diabetes
Total features: 8, Dropped Categorical features: 0, Remaining features: 8
Configuring DataLoader for CPU.
Configuring DataLoader for CPU.
Initialized QuantumNN on device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:59: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] Saved timing-only results to /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/utils/benchmarks/timing_results/BenchRL-QAS-TPPO/diabetes/seed_0/8qubits/final_results.json


In [24]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd

BENCHMARKS_DIR = Path.cwd()
TIMING_ROOT = BENCHMARKS_DIR / "timing_results"

METHOD_DIRS = {
    "QuantumDARTS": "QuantumDARTS",
    "TF-QAS": "TF-QAS",
    "BenchRL-QAS (TPPO)": "BenchRL-QAS-TPPO",
}

TARGET_ROWS = [
    ("iris", 4),
    ("wine", 4),
    ("wine", 6),
    ("wine", 8),
    ("diabetes", 4),
    ("diabetes", 6),
    ("diabetes", 8),
]

DATASET_LABELS = {
    "iris": "Iris",
    "wine": "Wine",
    "diabetes": "Diabetes",
}


def load_all_timing_results():
    rows = []

    for method_label, method_dirname in METHOD_DIRS.items():
        method_root = TIMING_ROOT / method_dirname
        if not method_root.exists():
            print(f"[WARN] Missing method folder: {method_root}")
            continue

        for dataset_dir in method_root.iterdir():
            if not dataset_dir.is_dir():
                continue

            dataset = dataset_dir.name.lower()

            for seed_dir in dataset_dir.glob("seed_*"):
                seed_str = seed_dir.name.replace("seed_", "")
                try:
                    seed = int(seed_str)
                except ValueError:
                    seed = seed_str

                for qubit_dir in seed_dir.glob("*qubits"):
                    result_fp = qubit_dir / "final_results.json"
                    if not result_fp.exists():
                        continue

                    try:
                        qubits = int(qubit_dir.name.replace("qubits", ""))
                    except ValueError:
                        continue

                    with open(result_fp, "r") as f:
                        data = json.load(f)

                    runtime = data.get("runtime_table_sec", {})
                    rows.append({
                        "method": method_label,
                        "dataset": dataset,
                        "qubits": qubits,
                        "seed": seed,
                        "search": runtime.get("search"),
                        "train": runtime.get("train"),
                        "total": runtime.get("total"),
                    })

    return pd.DataFrame(rows)


def summarise_timings(df):
    if df.empty:
        return df

    grouped = (
        df.groupby(["method", "dataset", "qubits"], as_index=False)
          .agg(
              search_mean=("search", "mean"),
              search_std=("search", "std"),
              train_mean=("train", "mean"),
              train_std=("train", "std"),
              total_mean=("total", "mean"),
              total_std=("total", "std"),
              n=("seed", "count"),
          )
    )
    return grouped


def fmt(x):
    if pd.isna(x):
        return "[\\;]"
    return f"{x:.2f}"


def get_cell(summary_df, method, dataset, qubits, metric):
    sub = summary_df[
        (summary_df["method"] == method) &
        (summary_df["dataset"] == dataset) &
        (summary_df["qubits"] == qubits)
    ]
    if sub.empty:
        return "[\\;]"
    return fmt(sub.iloc[0][f"{metric}_mean"])


def build_latex_rows(summary_df):
    rows = []

    for dataset, qubits in TARGET_ROWS:
        dataset_label = DATASET_LABELS[dataset]

        wi = ["[\\;]", "[\\;]", "[\\;]"]  # still placeholders
        tf = [
            get_cell(summary_df, "TF-QAS", dataset, qubits, "search"),
            get_cell(summary_df, "TF-QAS", dataset, qubits, "train"),
            get_cell(summary_df, "TF-QAS", dataset, qubits, "total"),
        ]
        qd = [
            get_cell(summary_df, "QuantumDARTS", dataset, qubits, "search"),
            get_cell(summary_df, "QuantumDARTS", dataset, qubits, "train"),
            get_cell(summary_df, "QuantumDARTS", dataset, qubits, "total"),
        ]
        rl = [
            get_cell(summary_df, "BenchRL-QAS (TPPO)", dataset, qubits, "search"),
            get_cell(summary_df, "BenchRL-QAS (TPPO)", dataset, qubits, "train"),
            get_cell(summary_df, "BenchRL-QAS (TPPO)", dataset, qubits, "total"),
        ]

        line = (
            f"{dataset_label} & {qubits} "
            f"& {wi[0]} & {wi[1]} & {wi[2]} "
            f"& {tf[0]} & {tf[1]} & {tf[2]} "
            f"& {qd[0]} & {qd[1]} & {qd[2]} "
            f"& {rl[0]} & {rl[1]} & {rl[2]} \\\\"
        )
        rows.append(line)

        if (dataset, qubits) == ("iris", 4):
            rows.append(r"\midrule")
        elif (dataset, qubits) == ("wine", 8):
            rows.append(r"\midrule")

    return rows

In [25]:
df_raw = load_all_timing_results()
df_summary = summarise_timings(df_raw)

print("Raw timing rows:")
display(df_raw.sort_values(["method", "dataset", "qubits", "seed"]))

print("\nSummary:")
display(df_summary.sort_values(["dataset", "qubits", "method"]))

Raw timing rows:


,method,dataset,qubits,seed,search,train,total
68,BenchRL-QAS (TPPO),diabetes,2,0,3081.032123,62.389060,3143.421183
70,BenchRL-QAS (TPPO),diabetes,4,0,3941.260878,137.526251,4078.787129
69,BenchRL-QAS (TPPO),diabetes,6,0,3575.424443,29.278730,3604.703173
71,BenchRL-QAS (TPPO),diabetes,8,0,6163.178594,38.405320,6201.583914
62,BenchRL-QAS (TPPO),iris,2,0,410.126956,16.061725,426.188681
63,BenchRL-QAS (TPPO),iris,4,0,583.671528,36.383389,620.054917
64,BenchRL-QAS (TPPO),wine,2,0,747.199762,20.173849,767.373611
66,BenchRL-QAS (TPPO),wine,4,0,636.907313,29.175819,666.083132
65,BenchRL-QAS (TPPO),wine,6,0,739.169842,6.467650,745.637491
67,BenchRL-QAS (TPPO),wine,8,0,757.960049,9.633186,767.593235



Summary:


,method,dataset,qubits,search_mean,search_std,train_mean,train_std,total_mean,total_std,n
0,BenchRL-QAS (TPPO),diabetes,2,3081.032123,NaN,62.389060,NaN,3143.421183,NaN,1
10,QuantumDARTS,diabetes,2,593.546828,NaN,419.523108,NaN,1013.069936,NaN,1
20,TF-QAS,diabetes,2,32851.550723,NaN,607.265975,NaN,33458.816698,NaN,1
1,BenchRL-QAS (TPPO),diabetes,4,3941.260878,NaN,137.526251,NaN,4078.787129,NaN,1
11,QuantumDARTS,diabetes,4,1009.332801,73.438822,564.454536,84.303584,1573.787337,144.840714,5
21,TF-QAS,diabetes,4,43010.051066,439.541411,837.181982,53.679785,43847.233047,406.981001,3
2,BenchRL-QAS (TPPO),diabetes,6,3575.424443,NaN,29.278730,NaN,3604.703173,NaN,1
12,QuantumDARTS,diabetes,6,1699.062886,122.392765,799.175650,122.774226,2498.238536,227.164794,5
22,TF-QAS,diabetes,6,63832.429642,1226.041781,1216.772118,68.098168,65049.201760,1289.689444,3
3,BenchRL-QAS (TPPO),diabetes,8,6163.178594,NaN,38.405320,NaN,6201.583914,NaN,1


In [14]:
latex_rows = build_latex_rows(df_summary)
print("\n".join(latex_rows))

Iris & 4 & [\;] & [\;] & [\;] & 9263.70 & 168.55 & 9432.25 & 949.71 & 109.53 & 1059.24 & 583.67 & 36.38 & 620.05 \\
\midrule
Wine & 4 & [\;] & [\;] & [\;] & 10752.69 & 202.98 & 10955.66 & 862.49 & 131.36 & 993.85 & 636.91 & 29.18 & 666.08 \\
Wine & 6 & [\;] & [\;] & [\;] & 15212.73 & 272.79 & 15485.52 & 1465.46 & 185.53 & 1650.99 & 739.17 & 6.47 & 745.64 \\
Wine & 8 & [\;] & [\;] & [\;] & 22572.57 & 419.23 & 22991.79 & 2320.85 & 273.99 & 2594.84 & 757.96 & 9.63 & 767.59 \\
\midrule
Diabetes & 4 & [\;] & [\;] & [\;] & 43010.05 & 837.18 & 43847.23 & 1009.33 & 564.45 & 1573.79 & 3941.26 & 137.53 & 4078.79 \\
Diabetes & 6 & [\;] & [\;] & [\;] & 63832.43 & 1216.77 & 65049.20 & 1699.06 & 799.18 & 2498.24 & 3575.42 & 29.28 & 3604.70 \\
Diabetes & 8 & [\;] & [\;] & [\;] & 94474.02 & 1799.23 & 96273.25 & 2693.05 & 1183.86 & 3876.91 & 6163.18 & 38.41 & 6201.58 \\


In [26]:
pd.set_option("display.max_rows", None)

df_raw["search_min"] = df_raw["search"] / 60
df_raw["train_min"] = df_raw["train"] / 60
df_raw["total_min"] = df_raw["total"] / 60

In [30]:
df_raw[(df_raw['dataset'] == 'wine') & (df_raw['qubits'] == 8)]

,method,dataset,qubits,seed,search,train,total,search_min,train_min,total_min
8,QuantumDARTS,wine,8,2,2166.502700,232.035364,2398.538064,36.108378,3.867256,39.975634
11,QuantumDARTS,wine,8,4,2308.162046,326.604416,2634.766462,38.469367,5.443407,43.912774
14,QuantumDARTS,wine,8,3,2519.156896,305.620428,2824.777324,41.985948,5.093674,47.079622
17,QuantumDARTS,wine,8,1,2164.315800,228.801849,2393.117649,36.071930,3.813364,39.885294
21,QuantumDARTS,wine,8,0,2446.091516,276.884026,2722.975542,40.768192,4.614734,45.382926
44,TF-QAS,wine,8,2,22862.538095,404.403740,23266.941835,381.042302,6.740062,387.782364
47,TF-QAS,wine,8,1,21877.005836,456.334501,22333.340337,364.616764,7.605575,372.222339
51,TF-QAS,wine,8,0,22978.152175,396.940018,23375.092193,382.969203,6.615667,389.584870
67,BenchRL-QAS (TPPO),wine,8,0,757.960049,9.633186,767.593235,12.632667,0.160553,12.793221
